# Lesson 4: Dynamic Programming (Knapsack & LCS)

**Source Reference:** [Data Structures and Algorithms in Python (freeCodeCamp / Jovian)](https://www.youtube.com/watch?v=pkYVOmU3MgA)

## Problem 1: The 0/1 Knapsack Problem
> **Question:** Given $N$ elements, each of which has a weight and a profit, determine the maximum profit that can be obtained by selecting a subset of the elements weighing no more than a given capacity $W$.

### Input and Output Formats
* **Input:** `weights` (list of integers), `profits` (list of integers), `capacity` (integer).
* **Output:** `max_profit` (integer).

In [1]:
# Test cases mapping standard sets, exact capacity bounds, and zero-fit scenarios
knapsack_tests = [
    {'input': {'capacity': 165, 'weights': [23, 31, 29, 44, 53, 38, 63, 85, 89, 82], 'profits': [92, 57, 49, 68, 60, 43, 67, 84, 87, 72]}, 'output': 309},
    {'input': {'capacity': 3, 'weights': [4, 5, 6], 'profits': [1, 2, 3]}, 'output': 0}, # Cannot fit anything
    {'input': {'capacity': 4, 'weights': [4, 5, 1], 'profits': [1, 2, 3]}, 'output': 3}, 
    {'input': {'capacity': 15, 'weights': [4, 5, 1, 3, 2, 5], 'profits': [2, 3, 1, 5, 4, 7]}, 'output': 19}
]

## Approach 1 & 2: Recursion and Memoization

### Core Mechanic: The "Take It or Leave It" Decision
For every single item in the list, we have two choices:
1. **Leave it:** Move to the next item. The capacity stays the same.
2. **Take it:** Add its profit to our total, and subtract its weight from our remaining capacity. Move to the next item.
We recursively branch out these two choices for *every* item and return the maximum path. 

* **Time Complexity (Pure Recursion):** $O(2^N)$. This creates a massive, branching decision tree that calculates the exact same scenarios multiple times.
* **Time Complexity (Memoization):** $O(N \cdot W)$. By storing the result of a `(capacity, index)` state in a dictionary, we never calculate the same sub-problem twice.

In [2]:
def max_profit_recursive(weights, profits, capacity, idx=0):
    """Brute-force O(2^N) recursive decision tree."""
    # Base case: We ran out of items to check
    if idx == len(weights):
        return 0
    
    # If the current item is too heavy, we MUST leave it
    if weights[idx] > capacity: 
        return max_profit_recursive(weights, profits, capacity, idx + 1)
    
    # Otherwise, calculate the max between Taking it and Leaving it
    else:
        option_leave = max_profit_recursive(weights, profits, capacity, idx + 1)
        option_take = profits[idx] + max_profit_recursive(weights, profits, capacity - weights[idx], idx + 1)
        return max(option_leave, option_take)

def max_profit_memo(weights, profits, capacity):
    """Optimized O(N * W) Top-Down Dynamic Programming."""
    memo = {}
    
    def recurse(cap, idx=0):
        key = (cap, idx)
        if key in memo:
            return memo[key]
        
        if idx == len(weights):
            memo[key] = 0
        elif weights[idx] > cap: 
            memo[key] = recurse(cap, idx + 1)
        else:
            option_leave = recurse(cap, idx + 1)
            option_take = profits[idx] + recurse(cap - weights[idx], idx + 1)
            memo[key] = max(option_leave, option_take)
            
        return memo[key]
    
    return recurse(capacity)

## Approach 3: Tabulation (Bottom-Up DP)
Instead of recursion and call stacks, we build a 2D array (a table) representing `items` on one axis and `capacities` on the other. We systematically fill this grid from $0$ up to $W$.

### Complexity Analysis
* **Time Complexity:** $O(N \cdot W)$. We iterate exactly once through an $N \times W$ matrix.
* **Space Complexity:** $O(N \cdot W)$ to store the matrix in memory.

In [3]:
def max_profit_dp(weights, profits, capacity):
    """Iterative Bottom-Up Dynamic Programming implementation."""
    n = len(weights)
    
    # Create a 2D table of 0s. Size is (n+1) rows by (capacity+1) columns
    table = [[0 for _ in range(capacity + 1)] for _ in range(n + 1)]
    
    for i in range(n):
        for c in range(1, capacity + 1):
            if weights[i] > c:
                # Item is too heavy for current capacity 'c'. Carry down the previous best.
                table[i+1][c] = table[i][c]
            else:
                # Compare leaving the item vs taking the item
                option_leave = table[i][c] 
                option_take = profits[i] + table[i][c - weights[i]]
                table[i+1][c] = max(option_leave, option_take)
                
    # The final answer mathematically propagates to the bottom-right corner
    return table[-1][-1]

## Problem 2: Longest Common Subsequence (LCS)
> **Question:** Write a function to find the length of the longest common subsequence between two sequences. (A subsequence allows deleting characters, but maintains the original deterministic ordering). E.g., The LCS of "serendipitous" and "precipitation" is "reipito" (Length: 7).

### Input and Output Formats
* **Input:** `seq1` (string/list), `seq2` (string/list).
* **Output:** `len_lcs` (integer).

In [4]:
lcs_tests = [
    {'input': {'seq1': 'serendipitous', 'seq2': 'precipitation'}, 'output': 7},
    {'input': {'seq1': [1, 3, 5, 6, 7, 2, 5, 2, 3], 'seq2': [6, 2, 4, 7, 1, 5, 6, 2, 3]}, 'output': 5},
    {'input': {'seq1': 'longest', 'seq2': 'stone'}, 'output': 3},
    {'input': {'seq1': 'asdfwevad', 'seq2': 'opkpoiklklj'}, 'output': 0}, # No match
    {'input': {'seq1': 'dense', 'seq2': 'condensed'}, 'output': 5},        # One is subsequence of the other
    {'input': {'seq1': '', 'seq2': 'opkpoiklklj'}, 'output': 0},           # Empty string
]

## LCS Implementation (Recursion, Memoization, & Tabulation)

### Core Mechanic: The Shift and Compare
We use two index pointers to look at the **current elements** of both sequences.
1. **If they match:** Add `1` to our counter, and move BOTH pointers forward by one step.
2. **If they don't match:** We branch. What is the max length if we skip the current letter of Sequence A? What if we skip the current letter of Sequence B?
This recursive tree is identical in shape to the Knapsack problem, meaning we can optimize it using either **Top-Down Memoization** (a dictionary cache) or **Bottom-Up Tabulation** (a 2D grid).

### Complexity Analysis (Optimized DP)
* **Time Complexity:** $O(N_1 \cdot N_2)$. Both Memoization and Tabulation completely prune the $O(2^{N+M})$ recursive tree, ensuring we calculate each coordinate pair exactly once.
* **Space Complexity:** $O(N_1 \cdot N_2)$ to store the Cache dictionary or the Tabulation grid.

In [5]:
def lcs_recursive(seq1, seq2, idx1=0, idx2=0):
    """Brute force O(2^(N+M)) recursive tree."""
    # Base Case: Reached the end of either sequence
    if idx1 == len(seq1) or idx2 == len(seq2):  
        return 0
    
    # If characters match, lock it in (+1) and advance both pointers
    if seq1[idx1] == seq2[idx2]:
        return 1 + lcs_recursive(seq1, seq2, idx1 + 1, idx2 + 1)
    # If they don't match, find the max of advancing A vs advancing B
    else:
        return max(
            lcs_recursive(seq1, seq2, idx1 + 1, idx2),
            lcs_recursive(seq1, seq2, idx1, idx2 + 1)
        )
        
        
def lcs_memo(seq1, seq2):
    """Optimized O(N * M) Top-Down Dynamic Programming (Memoization)."""
    # 1. Initialize the cache to store already computed coordinate states
    memo = {}
    
    def recurse(idx1=0, idx2=0):
        # 2. Create a unique key for the current position
        key = (idx1, idx2)
        
        # 3. Cache Check: If we have solved this exact state before, skip the math
        if key in memo:
            return memo[key]
            
        # 4. Base Case: Reached the end of either sequence
        if idx1 == len(seq1) or idx2 == len(seq2):
            memo[key] = 0
            
        # 5. If characters match, lock it in (+1) and advance both pointers
        elif seq1[idx1] == seq2[idx2]:
            memo[key] = 1 + recurse(idx1 + 1, idx2 + 1)
            
        # 6. If they don't match, find the max of advancing A vs advancing B
        else:
            memo[key] = max(
                recurse(idx1 + 1, idx2),
                recurse(idx1, idx2 + 1)
            )
            
        # 7. Return the newly calculated and cached result
        return memo[key]
        
    return recurse(0, 0)


def lcs_dynamic(seq1, seq2):
    """Optimized O(N * M) Iterative Bottom-Up Tabulation."""
    n1, n2 = len(seq1), len(seq2)
    
    # Create an (n1 + 1) by (n2 + 1) matrix initialized with 0s
    table = [[0 for _ in range(n2 + 1)] for _ in range(n1 + 1)]
    
    for i in range(n1):
        for j in range(n2):
            if seq1[i] == seq2[j]:
                # Match found: take diagonal previous best and add 1
                table[i+1][j+1] = 1 + table[i][j]
            else:
                # No match: pull the best total from either above or to the left
                table[i+1][j+1] = max(table[i][j+1], table[i+1][j])
                
    return table[-1][-1]